<a href="https://colab.research.google.com/github/postolanastasia3-stack/A-B-test-conversion-analysis/blob/main/Portfolio_Project_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import auth
from google.colab import files

auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

PROJECT = "data-analytics-mate"
client  = bigquery.Client(project=PROJECT)
print("Auth OK")

Auth OK


In [ ]:
query = """
WITH session_info AS (
  SELECT
        s.date, s.ga_session_id,
        sp.country, sp.device, sp.continent, sp.channel,
        ab.test, ab.test_group
  FROM `DA.ab_test`        AS ab
  JOIN `DA.session`        AS s  ON ab.ga_session_id = s.ga_session_id
  JOIN `DA.session_params` AS sp ON sp.ga_session_id = ab.ga_session_id
),
events AS (
  SELECT si.date, si.country, si.device, si.continent, si.channel,
         si.test, si.test_group, ep.event_name,
         COUNT(ep.ga_session_id) AS event_cnt
  FROM `DA.event_params` AS ep
  JOIN session_info AS si ON ep.ga_session_id = si.ga_session_id
  GROUP BY 1,2,3,4,5,6,7,8
),
session AS (
  SELECT date, country, device, continent, channel, test, test_group,
         COUNT(DISTINCT ga_session_id) AS session_cnt
  FROM session_info
  GROUP BY 1,2,3,4,5,6,7
),
account AS (
  SELECT si.date, si.country, si.device, si.continent, si.channel,
         si.test, si.test_group,
         COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
  FROM `DA.account_session` AS acs
  JOIN session_info AS si ON acs.ga_session_id = si.ga_session_id
  GROUP BY 1,2,3,4,5,6,7
)

SELECT date, country, device, continent, channel, test, test_group,
       'add_payment_info'  AS event_name, event_cnt AS value
FROM events WHERE event_name = 'add_payment_info'
UNION ALL
SELECT date, country, device, continent, channel, test, test_group,
       'add_shipping_info' AS event_name, event_cnt AS value
FROM events WHERE event_name = 'add_shipping_info'
UNION ALL
SELECT date, country, device, continent, channel, test, test_group,
       'begin_checkout'    AS event_name, event_cnt AS value
FROM events WHERE event_name = 'begin_checkout'
UNION ALL
SELECT date, country, device, continent, channel, test, test_group,
       'new_accounts'      AS event_name, new_account_cnt AS value
FROM account
UNION ALL
SELECT date, country, device, continent, channel, test, test_group,
       'session'           AS event_name, session_cnt AS value
FROM session
"""

df = client.query(query).to_dataframe()
print(f"Rows loaded: {len(df):,}")
print("Events in data:", df["event_name"].unique())
print("Test groups:   ", df["test_group"].unique())


Rows loaded: 161,567
Events in data: ['add_shipping_info' 'begin_checkout' 'add_payment_info' 'new_accounts'
 'session']
Test groups:    <IntegerArray>
[2, 1]
Length: 2, dtype: Int64


In [ ]:
# 4 цільові метрики
TARGET_METRICS = [
    "add_payment_info",
    "add_shipping_info",
    "begin_checkout",
    "new_accounts",
]
DENOMINATOR = "session"

CONTROL_GROUP = 1
VARIANT_GROUP = 2

print("Config set:", TARGET_METRICS)

Config set: ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_accounts']


In [ ]:
#PIVOT
needed_events = TARGET_METRICS + [DENOMINATOR]
df_filtered = df[df["event_name"].isin(needed_events)]

pivot = (
    df_filtered
    .groupby(["test", "test_group", "event_name"])["value"]
    .sum()
    .unstack("event_name")
    .reset_index()
    .fillna(0)
)

# Які з цільових метрик реально є в даних
available_metrics = [m for m in TARGET_METRICS if m in pivot.columns]

print(f"Pivot shape: {pivot.shape}")
print(f"Available metrics: {available_metrics}")
display(pivot.head())

Pivot shape: (8, 7)
Available metrics: ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_accounts']


event_name,test,test_group,add_payment_info,add_shipping_info,begin_checkout,new_accounts,session
0,1,1,1988,3034,3784,3823,45362
1,1,2,2229,3221,4021,3681,45193
2,2,1,2344,3480,4262,4165,50637
3,2,2,2409,3510,4313,4184,50244
4,3,1,3623,5298,9532,5856,70047


In [ ]:
#  Z-TEST ФУНКЦІЯ

def z_test_proportion(success_a, n_a, success_b, n_b):
    """Two-proportion z-test. Повертає (z_stat, p_value)."""
    if n_a == 0 or n_b == 0:
        return np.nan, np.nan
    p_pool = (success_a + success_b) / (n_a + n_b)
    se     = np.sqrt(p_pool * (1 - p_pool) * (1/n_a + 1/n_b))
    if se == 0:
        return np.nan, np.nan
    z     = (success_b/n_b - success_a/n_a) / se
    p_val = 2 * (1 - stats.norm.cdf(abs(z)))
    return round(z, 9), round(p_val, 9)

print("z_test_proportion defined")

z_test_proportion defined


In [ ]:
# Розрахунку значущості

results = []

for test_num, grp in pivot.groupby("test"):
    control = grp[grp["test_group"] == CONTROL_GROUP]
    variant = grp[grp["test_group"] == VARIANT_GROUP]

    if control.empty or variant.empty:
        continue

    n_control = control[DENOMINATOR].sum()
    n_variant = variant[DENOMINATOR].sum()

    # цикл по метриках — без статичного закріплення кількості
    for metric in available_metrics:
        num_control = control[metric].sum()
        num_variant = variant[metric].sum()

        rate_control = num_control / n_control if n_control > 0 else np.nan
        rate_variant = num_variant / n_variant if n_variant > 0 else np.nan
        metric_change = (rate_variant - rate_control) / rate_control * 100 if rate_control and rate_control > 0 else np.nan

        z, p = z_test_proportion(num_control, n_control, num_variant, n_variant)

        results.append({
            "test_number":        int(test_num),
            "metric":             metric,
            "numerator_event":    metric,
            "denominator_event":  DENOMINATOR,
            "numerator_control":  int(num_control),
            "denominator_control": int(n_control),
            "conversion_rate_control": rate_control,
            "numerator_variant":  int(num_variant),
            "denominator_variant": int(n_variant),
            "conversion_rate_variant": rate_variant,
            "metric_change":      metric_change,
            "z_stat":             z,
            "p_value":            p,
            "significant":        p < 0.05 if not np.isnan(p) else False,
        })

results_df = pd.DataFrame(results).sort_values(["test_number", "metric"]).reset_index(drop=True)
print(f"Done: {len(results_df)} rows")


Done: 16 rows


In [ ]:
# Запуск
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.9f}".format)

display(results_df)

,test_number,metric,numerator_event,denominator_event,numerator_control,denominator_control,conversion_rate_control,numerator_variant,denominator_variant,conversion_rate_variant,metric_change,z_stat,p_value,significant
0,1,add_payment_info,add_payment_info,session,1988,45362,0.043825228,2229,45193,0.049321798,12.542021318,3.924884023,0.000086772,True
1,1,add_shipping_info,add_shipping_info,session,3034,45362,0.066884176,3221,45193,0.071272100,6.560480713,2.603571455,0.009225803,True
2,1,begin_checkout,begin_checkout,session,3784,45362,0.083417839,4021,45193,0.088973956,6.660586643,2.978782962,0.002893957,True
3,1,new_accounts,new_accounts,session,3823,45362,0.084277589,3681,45193,0.081450667,-3.354298646,-1.542882615,0.122859260,False
4,2,add_payment_info,add_payment_info,session,2344,50637,0.046290262,2409,50244,0.047946023,3.576910717,1.240993925,0.214607993,False
5,2,add_shipping_info,add_shipping_info,session,3480,50637,0.068724451,3510,50244,0.069859088,1.650994869,0.709556865,0.477978977,False
6,2,begin_checkout,begin_checkout,session,4262,50637,0.084167703,4313,50244,0.085841095,1.988164020,0.952898041,0.340641733,False
7,2,new_accounts,new_accounts,session,4165,50637,0.082252108,4184,50244,0.083273625,1.241933602,0.588793473,0.555999825,False
8,3,add_payment_info,add_payment_info,session,3623,70047,0.051722415,3697,70439,0.052485129,1.474629573,0.643172401,0.520112240,False
9,3,add_shipping_info,add_shipping_info,session,5298,70047,0.075634931,5188,70439,0.073652380,-2.621210513,-1.413727145,0.157442032,False


Посилання на дашборд: https://public.tableau.com/app/profile/anastasiia.postol/viz/ABtest_17798856383550/Dashboard2?publish=yes

Сsv:

In [ ]:
results_df.to_csv("ab_test_results.csv", index=False)
files.download("ab_test_results.csv")

NameError: name 'results_df' is not defined